In [1]:
# [Cell 1] - Import Library & Helper Function
import pandas as pd
import numpy as np
import re
import joblib
import os

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

# Fungsi Pembersih Timestamp
def parse_hours(val):
    if pd.isna(val): return 0.0
    s = str(val)
    try: return float(s)
    except: pass
    match = re.search(r'(\d+)$', s)
    return float(match.group(1)) if match else 0.0

# [Cell 2] - Cleaning & Feature Engineering
df = pd.read_csv('./Delivery_Logistics.csv')

df['delivery_time_hours'] = df['delivery_time_hours'].apply(parse_hours)
df['expected_time_hours'] = df['expected_time_hours'].apply(parse_hours)

# Feature Engineering
df['efficiency_ratio'] = df['delivery_time_hours'] / (df['expected_time_hours'] + 1e-5)
df['tenure'] = 12

print("Data Setelah Pembersihan & Feature Engineering:")
df[['delivery_time_hours', 'expected_time_hours', 'efficiency_ratio']].head()

# [Cell 3] - Pipeline Preprocessing (Scaling & Encoding)
X = df.drop(columns=['delivery_status', 'delivery_id', 'is_delayed', 'delayed'], errors='ignore')
y = df['delivery_status'].map({'delivered': 0, 'delayed': 1, 'failed': 2})

num_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_features = X.select_dtypes(include=['object', 'category']).columns.tolist()

preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), num_features),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_features)
])

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

X_train_transformed = preprocessor.fit_transform(X_train)
X_test_transformed = preprocessor.transform(X_test)

print(f"✅ Dimensi Data Latih setelah Preprocessing: {X_train_transformed.shape}")

# [Cell 4] - Simpan Preprocessor
output_dir = './models'
os.makedirs(output_dir, exist_ok=True)
joblib.dump(preprocessor, os.path.join(output_dir, 'preprocessor.pkl'))
print(f"✅ Preprocessor berhasil disimpan di '{os.path.join(output_dir, 'preprocessor.pkl')}'")

Data Setelah Pembersihan & Feature Engineering:
✅ Dimensi Data Latih setelah Preprocessing: (20000, 47)
✅ Preprocessor berhasil disimpan di './models/preprocessor.pkl'
